# ЛР2. Обучение GPT-like модели
GPT-декодер с нуля (PyTorch+Lightning), данные/токенизатор из ЛР1.

In [1]:
import os, sys
# ROOT задаётся драйвером на Kaggle (/kaggle/working); локально — текущая папка проекта
ROOT = os.environ.get('LAB2_ROOT', os.path.abspath('..'))
sys.path.insert(0, ROOT); os.chdir(ROOT)
import torch, lightning as L
from omegaconf import OmegaConf
from src.utils import load_config, env
print('device cuda:', torch.cuda.is_available())

device cuda: True


## Конфиг (всё из YAML; SMOKE — быстрый прогон для проверки пайплайна)

In [2]:
cfg = load_config(os.environ.get('LAB2_CONFIG','configs/model.yaml'))
SMOKE = os.environ.get('LAB2_SMOKE','0') == '1'
if SMOKE:
    cfg.data.max_documents = 3000
    cfg.model.n_layers, cfg.model.d_model, cfg.model.d_ff = 4, 256, 1024
    cfg.optim.max_steps, cfg.optim.warmup_steps = 150, 30
    cfg.trainer.val_check_interval = 75
print(OmegaConf.to_yaml(cfg))

model:
  vocab_size: 32000
  max_seq_len: 512
  d_model: 512
  n_layers: 8
  n_heads: 8
  d_ff: 2048
  dropout: 0.1
  tie_weights: true
  pad_token_id: 0
data:
  dataset: wikitext
  bpe_tokenizer_path: /kaggle/input/datasets/ekaterinabulanova/lab2-tokenizer/bpe_tokenizer.json
  target_len: 512
  batch_size: 32
  num_workers: 2
  val_fraction: 0.05
  max_documents: null
logging:
  clearml_project: mnna-lab2
  clearml_task: gpt-wikitext
optim:
  lr: 0.0003
  weight_decay: 0.1
  betas:
  - 0.9
  - 0.95
  warmup_steps: 1000
  max_steps: 40000
  min_lr_ratio: 0.1
  grad_clip: 1.0
  target_perplexity: 30
trainer:
  max_epochs: -1
  precision: 16-mixed
  compile: true
  accumulate_grad_batches: 1
  val_check_interval: 1000
  log_every_n_steps: 50
  track_layer_grad_norms: false
  max_time: 00:08:30:00
generate:
  prompt: The history of
  max_new_tokens: 100
  temperature: 0.8
  top_k: 50



## Данные: wikitext + BPE из ЛР1 -> packed-батчи

In [3]:
from src.data.datamodule import LMDataModule
dm = LMDataModule(cfg); dm.setup()
cfg.model.vocab_size = dm.vocab_size
print('vocab:', dm.vocab_size, '| train batches:', len(dm.train_ds), '| val batches:', len(dm.val_ds))
b = dm.train_ds[0]; print('input_ids', b['input_ids'].shape, '| segment_ids уникальные:', b['segment_ids'].unique().tolist()[:6])

README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

vocab: 32000 | train batches: 162752 | val batches: 8565
input_ids torch.Size([512]) | segment_ids уникальные: [1]


## Модель

In [4]:
from src.training.lightning_module import GPTLitModule
model = GPTLitModule(cfg, dm.vocab_size)
print('параметров:', sum(p.numel() for p in model.parameters())/1e6, 'M')

параметров: 41.603072 M


## Обучение (ClearML/TensorBoard, warmup+cosine, grad-clip, чекпоинты)

In [5]:
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from src.training.callbacks import GradNormCallback
try:
    from clearml import Task; Task.init(project_name=cfg.logging.clearml_project, task_name=cfg.logging.clearml_task)
except Exception as e:
    print('ClearML off:', e)
ckpt_dir = env('CHECKPOINT_DIR','./checkpoints')
ckpt = ModelCheckpoint(dirpath=ckpt_dir, monitor='val_perplexity', mode='min', save_top_k=1, save_last=True, filename='gpt-{step}-{val_perplexity:.2f}')
trainer = L.Trainer(max_steps=cfg.optim.max_steps, precision=cfg.trainer.precision,
    accumulate_grad_batches=cfg.trainer.accumulate_grad_batches, val_check_interval=cfg.trainer.val_check_interval,
    log_every_n_steps=cfg.trainer.log_every_n_steps, max_time=cfg.trainer.get('max_time'), gradient_clip_val=cfg.optim.grad_clip, gradient_clip_algorithm='norm',
    logger=TensorBoardLogger(ckpt_dir, name='tb'), callbacks=[ckpt, LearningRateMonitor('step'), GradNormCallback(cfg.trainer.track_layer_grad_norms)])
trainer.fit(model, dm, ckpt_path=os.environ.get('RESUME_CKPT') or None)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


Using 16bit Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Restored all states from the checkpoint at /kaggle/input/lab2-ckpt/last.ckpt


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/training_epoch_loop.py:224: You're resuming from a 
checkpoint that ended before the epoch ended and your dataloader is not resumable. This can cause unreliable 
results if further training is done. Consider using an end-of-epoch checkpoint or make your dataloader resumable by
implementing the `state_dict` / `load_state_dict` interface.

W0624 20:46:13.551000 61 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


## Перплексия на валидации

In [6]:
trainer.validate(model, dm)
print('val_perplexity =', float(trainer.callback_metrics.get('val_perplexity', float('nan'))))

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val_loss          │    3.3674192428588867     │
│      val_perplexity       │    29.003576278686523     │
└───────────────────────────┴───────────────────────────┘

val_perplexity = 29.003576278686523


## Генерация в режиме инференса

In [7]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file(cfg.data.bpe_tokenizer_path)
ids = torch.tensor([tok.encode(cfg.generate.prompt).ids], device=model.device)
out = model.generate(ids, max_new_tokens=cfg.generate.max_new_tokens, temperature=cfg.generate.temperature, top_k=cfg.generate.top_k)
print('PROMPT:', cfg.generate.prompt)
print('GEN   :', tok.decode(out[0].tolist()))

PROMPT: The history of
GEN   : The history of the Purcellus pony , which is now known as the Purcellus , is a taxon in the genus of the fossil of the antigrants of the Latin genus Cretaceae , which is a family species of mammals from the mycestors of the genus Cretaceae . The specimen was first described in the 1930s by Ancient Bologna , a former tax
